# Phase 2 QLoRA SFT — Qwen2.5-Coder-7B CPT-merged (Unsloth Colab)

Self-contained notebook (no external train script). **Fresh QLoRA** on the **CPT-merged domain base** with ChatML + assistant-only loss.

| Item | Value |
| --- | --- |
| Init | Drive CPT-merged if complete, else Hub `Aniket200325/coder-qwen25-coder-7b-cpt-merged` |
| Precision | **QLoRA 4-bit** (`load_in_4bit=True`) |
| Fresh LoRA | `r=32`, `alpha=64` (attn + MLP); **do not** continue CPT adapters |
| Data | Hub `Aniket200325/coder-sft-mix-v1` (`train` + `eval_securecodepairs`) |
| Epochs / Packing | **2 Epochs** | **Packing Off** (`packing=False`) |
| Seq / LR | **2048** / **2e-5** cosine |
| Timed ckpt | **60 min** full / **5 min** smoke → Drive `LATEST` |
| Resume | `RESUME="auto"` (full) from Drive `checkpoint-*` |
| Adapter Hub | `Aniket200325/coder-qwen25-coder-7b-sft-qlora-v1` (private) |
| Final merge | **Cell 22 only** — set `RUN_FINAL_MERGE=True` after last train session |

**Bans:** Instruct **weights** as init; continuing CPT LoRA; packing on; ChatML template off.

Smoke first (`SMOKE=True`). Jetson quant later. Run Cell 22 only when training is fully done.


### Installation


In [ ]:
%%capture
import os, re

# BEFORE any Unsloth import — avoid fast CDN hang + stats probes
os.environ["UNSLOTH_STABLE_DOWNLOADS"] = "1"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ.pop(k, None)
os.environ["PYTHONUNBUFFERED"] = "1"

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {
        "2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2", "2.11": "0.0.35",
    }.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps transformers
    !pip install --no-deps --upgrade "torchao>=0.16.0"


In [ ]:
import os
import torch, transformers
print("transformers", transformers.__version__, "torch", torch.__version__)
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"GPUs detected: {num_gpus}")
if num_gpus > 0:
    for i in range(num_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("  GPU: CPU")
print("Download env OK:", {
    k: os.environ.get(k) for k in (
        "UNSLOTH_STABLE_DOWNLOADS", "UNSLOTH_DISABLE_STATISTICS",
        "HF_HUB_DISABLE_XET", "HF_HUB_ENABLE_HF_TRANSFER",
    )
})


### Config + auth

Fill `HF_TOKEN` (or Colab Secret). Set `SMOKE = True` for a short dry-run (masking + packing gates); `False` for a full SFT session with **1h Drive checkpoints** + `RESUME=auto`. Keep `RUN_FINAL_MERGE = False` until the last training session.


In [ ]:
import os, json, time, shutil, math, gc, random
from pathlib import Path
from dataclasses import dataclass, asdict

# ── fill these ──────────────────────────────────────────────────────────────
HF_TOKEN = ""  # or Colab Secrets / userdata
USE_DRIVE = True
SMOKE = True  # True → short smoke; False → full SFT session
RUN_FINAL_MERGE = False  # Cell 22 gate; flip True only on last run
FORCE_MERGE = False  # Cell 22: overwrite existing merge out_dir

MODEL_HUB = "Aniket200325/coder-qwen25-coder-7b-cpt-merged"
MODEL_DRIVE = Path("/content/drive/MyDrive/coder-qwen25-coder-7b-cpt-merged")
DATASET = "Aniket200325/coder-sft-mix-v1"
HUB_ADAPTER_ID = "Aniket200325/coder-qwen25-coder-7b-sft-qlora-v1"
HUB_MERGED_SFT_ID = ""  # optional private Hub for Cell 22 merged BF16; empty = Drive only

MAX_SEQ_LEN = 2048
PACKING = False
LOAD_IN_4BIT = True
LORA_R = 32
LORA_ALPHA = 64
LR = 2e-5
SEED = 42

BATCH = 2 if SMOKE else 4
ACCUM = 4 if SMOKE else 8  # effective 8 smoke / 32 full; tune after smoke
MAX_STEPS = 30 if SMOKE else None  # None → num_train_epochs
NUM_TRAIN_EPOCHS = 1
EVAL_STEPS = 10 if SMOKE else 200
SAVE_STEPS = 15 if SMOKE else 500
LOGGING_STEPS = 1 if SMOKE else 10
CKPT_MINUTES = 5.0 if SMOKE else 60.0  # 1h timed saves (Phase 1 pattern)
WALL_CLOCK_HOURS = 11.5  # stop before Colab ~12h kill; 0 = disable
RESUME = "none" if SMOKE else "auto"

OUT_DIR = Path(
    "/content/ckpts/qwen25-coder-7b-phase2-sft-smoke"
    if SMOKE
    else "/content/ckpts/qwen25-coder-7b-phase2-sft"
)
MERGE_OUT_DIR = Path("/content/drive/MyDrive/coder-qwen25-coder-7b-sft-merged")
# ────────────────────────────────────────────────────────────────────────────

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        HF_TOKEN = ""
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or ""

assert HF_TOKEN, (
    "Set HF_TOKEN (Hugging Face token) — private Hub model + dataset require auth"
)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

DRIVE_CKPT = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_CKPT = Path(
            "/content/drive/MyDrive/coder-qwen25-coder-7b-phase2-sft-smoke"
            if SMOKE
            else "/content/drive/MyDrive/coder-qwen25-coder-7b-phase2-sft"
        )
        DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print(f"Notice: Google Drive unavailable ({e}). Continuing without Drive mirroring.")
        USE_DRIVE = False
        DRIVE_CKPT = None

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Throughput / Ampere knobs
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("SMOKE:", SMOKE, "| OUT:", OUT_DIR, "| DRIVE:", DRIVE_CKPT)
print(
    "batch", BATCH, "accum", ACCUM, "effective", BATCH * ACCUM,
    "| seq", MAX_SEQ_LEN, "| packing", PACKING,
    "| CKPT_MINUTES", CKPT_MINUTES, "| resume", RESUME,
)
print("wall_clock_stop_h", WALL_CLOCK_HOURS, "| RUN_FINAL_MERGE", RUN_FINAL_MERGE)
print("HUB_ADAPTER_ID", HUB_ADAPTER_ID)


### Load CPT-merged base (QLoRA 4-bit) + ChatML template

Prefer Drive merge if complete; else Hub snapshot. Weights stay the **merged Base** — attach Instruct **ChatML template only**, not Instruct weights.


In [ ]:
# Resolve CPT-merged domain base → prefetch → QLoRA load → ChatML template.
import os
os.environ.setdefault("UNSLOTH_STABLE_DOWNLOADS", "1")
os.environ.setdefault("UNSLOTH_DISABLE_STATISTICS", "1")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ.pop(k, None)

from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

def local_model_complete(path: Path) -> bool:
    if not path.is_dir() or not (path / "config.json").is_file():
        return False
    for pattern in ("*.safetensors", "pytorch_model*.bin", "model*.bin"):
        for p in path.glob(pattern):
            if p.is_file() and p.stat().st_size > 1_000_000:
                return True
    return False

def is_adapter_only_dir(path: Path) -> bool:
    """True if dir looks like PEFT adapters without a full base config+weights."""
    if not path.is_dir():
        return False
    has_adapter = (path / "adapter_config.json").is_file()
    has_full = local_model_complete(path)
    return has_adapter and not has_full

MODEL_CACHE = Path("/content/models") / MODEL_HUB.replace("/", "--")

# 1) Resolve model path
resolved = None
if USE_DRIVE and local_model_complete(MODEL_DRIVE):
    resolved = MODEL_DRIVE
    print("Using Drive CPT-merged base:", resolved)
elif local_model_complete(MODEL_CACHE):
    resolved = MODEL_CACHE
    print("Using cached Hub snapshot:", resolved)
else:
    if USE_DRIVE and MODEL_DRIVE.exists() and is_adapter_only_dir(MODEL_DRIVE):
        raise SystemExit(
            f"MODEL_DRIVE looks like adapters only (adapter_config.json, no full weights): {MODEL_DRIVE}\n"
            "Point at the CPT-merged BF16 base (run fine-tune/merge_cpt_lora_colab.py first), "
            "not Phase-1 final/ LoRA."
        )
    if MODEL_CACHE.exists():
        print("Incomplete local dir; removing", MODEL_CACHE)
        shutil.rmtree(MODEL_CACHE, ignore_errors=True)
    MODEL_CACHE.parent.mkdir(parents=True, exist_ok=True)
    print("Prefetching", MODEL_HUB, "→", MODEL_CACHE)
    try:
        snapshot_download(
            repo_id=MODEL_HUB,
            local_dir=str(MODEL_CACHE),
            token=HF_TOKEN,
            resume_download=True,
            max_workers=8,
        )
    except Exception as e:
        raise SystemExit(
            f"Could not load CPT-merged base from Drive or Hub.\n"
            f"Drive complete={local_model_complete(MODEL_DRIVE) if USE_DRIVE else False}, "
            f"Hub prefetch error: {e}\n"
            "Run fine-tune/merge_cpt_lora_colab.py first, or ensure Hub "
            f"{MODEL_HUB} is reachable with your token."
        ) from e
    if not local_model_complete(MODEL_CACHE):
        raise SystemExit(
            f"Prefetch incomplete: {MODEL_CACHE}. "
            "Merge not done — see fine-tune/merge_cpt_lora_colab.py."
        )
    gb = sum(p.stat().st_size for p in MODEL_CACHE.rglob("*") if p.is_file()) / 1e9
    print(f"Prefetch complete ({gb:.1f} GB)")
    resolved = MODEL_CACHE

assert resolved is not None and local_model_complete(resolved), (
    "No complete CPT-merged base. Run fine-tune/merge_cpt_lora_colab.py."
)

# 2) Hard lineage checks
if is_adapter_only_dir(resolved):
    raise SystemExit(
        f"Resolved path is adapter-only (CPT LoRA?), not a merged base: {resolved}"
    )
if "instruct" in str(resolved).lower() or "Instruct" in MODEL_HUB:
    print(
        "WARNING: path/id contains 'Instruct' — Phase 2 should init from CPT-merged "
        "Base weights + ChatML template only, not Instruct weights.",
        flush=True,
    )

MODEL_DIR = Path(resolved)
print("MODEL_DIR:", MODEL_DIR)

# 3) Load QLoRA (Multi-GPU compatible)
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
load_kwargs = dict(
    model_name=str(MODEL_DIR),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,  # auto bf16/fp16 compute
    load_in_4bit=LOAD_IN_4BIT,
    token=HF_TOKEN,
)
if num_gpus > 1 and "LOCAL_RANK" not in os.environ:
    load_kwargs["device_map"] = "auto"

try:
    model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
except TypeError:
    load_kwargs.pop("device_map", None)
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
    except TypeError:
        load_kwargs.pop("token", None)
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
        except TypeError:
            load_kwargs.pop("max_seq_length", None)
            model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)

# 4) Chat template (weights stay merged Base; template only)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
assert getattr(tokenizer, "chat_template", None), (
    "tokenizer.chat_template is empty after get_chat_template"
)
_dummy = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Say hi."},
    {"role": "assistant", "content": "Hello."},
]
_rendered = tokenizer.apply_chat_template(
    _dummy, tokenize=False, add_generation_prompt=False
)
assert "<|im_start|>assistant" in _rendered, (
    f"ChatML sanity failed — missing <|im_start|>assistant in:\n{_rendered[:400]}"
)
print("ChatML template OK; sample head:", repr(_rendered[:160]))

cfg = model.config
archs = list(getattr(cfg, "architectures", None) or [])
has_vision = hasattr(cfg, "vision_config")
print(
    "Loaded CPT-merged | tokenizer", type(tokenizer).__name__,
    f"| load_in_4bit={LOAD_IN_4BIT}",
)
print("architectures:", archs, "| vision_config:", has_vision)
assert not has_vision, "Model has vision_config — unexpected for this Phase 2 path."
assert not any(a.endswith("ForConditionalGeneration") for a in archs), (
    f"ConditionalGeneration architecture {archs} → wrong checkpoint."
)


### Fresh LoRA (do not load CPT adapters)

New PEFT scaffold only. On resume, Trainer reloads adapter + optimizer state from `checkpoint-*` after this cell.


In [ ]:
# Fresh LoRA — never PeftModel.from_pretrained(CPT adapters) here.
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
print("Fresh LoRA ready: r=", LORA_R, "alpha=", LORA_ALPHA, "| 4bit=", LOAD_IN_4BIT)
print("(Resume will overlay adapter/optimizer weights from checkpoint-* if RESUME=auto)")


<a name="Data"></a>
### Dataset

Hub `coder-sft-mix-v1`: conversational `messages` → ChatML `text`. Schema gate + smoke slice before full 136K.


In [ ]:
from datasets import load_dataset
from collections import Counter

REQUIRED_ROLES = ["system", "user", "assistant"]

def valid_messages(msgs) -> bool:
    if not isinstance(msgs, list) or len(msgs) < 3:
        return False
    roles = [m.get("role") for m in msgs[:3]]
    if roles != REQUIRED_ROLES:
        return False
    for m in msgs:
        c = m.get("content")
        if not isinstance(c, str) or not c.strip():
            return False
    return True

print("Loading", DATASET, "…")
try:
    raw = load_dataset(DATASET, token=HF_TOKEN)
except TypeError:
    raw = load_dataset(DATASET)

assert "train" in raw, f"Missing train split; got {list(raw.keys())}"
assert "eval_securecodepairs" in raw, (
    f"Missing eval_securecodepairs; got {list(raw.keys())}"
)
train_raw = raw["train"]
eval_raw = raw["eval_securecodepairs"]

cols = set(train_raw.column_names)
for need in ("messages", "source", "task"):
    assert need in cols, f"train missing column {need}; have {sorted(cols)}"

# Schema gate: first 200 + random 50
n = len(train_raw)
scan_idx = list(range(min(200, n)))
if n > 200:
    rng = random.Random(SEED)
    extra = rng.sample(range(200, n), k=min(50, n - 200))
    scan_idx.extend(extra)
bad = 0
for i in scan_idx:
    if not valid_messages(train_raw[i]["messages"]):
        bad += 1
        if bad <= 3:
            print("Bad row sample idx", i, train_raw[i]["messages"][:1])
assert bad == 0, f"Schema gate failed: {bad}/{len(scan_idx)} rows bad roles/contents"

def keep_row(example):
    return valid_messages(example.get("messages"))

_nproc = 2 if n > 1000 else None
train_ds = train_raw.filter(keep_row, num_proc=_nproc)
eval_ds = eval_raw.filter(keep_row, num_proc=1)

if SMOKE:
    train_ds = train_ds.select(range(min(256, len(train_ds))))
    eval_ds = eval_ds.select(range(min(32, len(eval_ds))))

def to_text(example):
    try:
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception:
        text = ""
    if not isinstance(text, str):
        text = ""
    return {"text": text}

_map_proc = 2 if len(train_ds) > 64 else None
train_ds = train_ds.map(to_text, num_proc=_map_proc)
eval_ds = eval_ds.map(to_text, num_proc=1)
train_ds = train_ds.filter(lambda ex: bool(ex["text"] and ex["text"].strip()))
eval_ds = eval_ds.filter(lambda ex: bool(ex["text"] and ex["text"].strip()))

task_counts = Counter(train_ds["task"]) if "task" in train_ds.column_names else {}
src_counts = Counter(train_ds["source"]) if "source" in train_ds.column_names else {}
print("train", len(train_ds), "| eval", len(eval_ds), "| SMOKE", SMOKE)
print("task counts:", dict(task_counts))
print("source counts:", dict(src_counts))
sample = train_ds[0]
print("sample task/source:", sample.get("task"), sample.get("source"))
print("text head:", repr(sample["text"][:240]))

train_dataset = train_ds
eval_dataset = eval_ds


<a name="Train"></a>
### Trainer + assistant-only mask + smoke gates

`packing=False` required. `train_on_responses_only` with Qwen markers. Label audit **aborts** before a long train. Timed Drive checkpoints every `CKPT_MINUTES`.


In [ ]:
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

@dataclass
class Phase2State:
    global_step: int = 0
    best_eval_loss: float = float("inf")
    last_eval_loss: float = float("nan")
    tok_approx: int = 0

def write_latest(root: Path, ckpt: Path):
    root.mkdir(parents=True, exist_ok=True)
    (root / "LATEST").write_text(str(ckpt.resolve()))

def read_latest(root: Path):
    f = root / "LATEST"
    if not f.exists():
        return None
    p = Path(f.read_text().strip())
    return p if p.exists() else None

def mirror_to_drive(src: Path, update_latest: bool = True):
    if DRIVE_CKPT is None:
        return
    dest = DRIVE_CKPT / src.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(src, dest, dirs_exist_ok=True)
    if update_latest:
        write_latest(DRIVE_CKPT, dest)
    print("Mirrored →", dest, flush=True)

def is_valid_resume_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    # Never resume from adapters-only final/
    if p.name == "final":
        return False
    return (p / "trainer_state.json").is_file()

def resolve_resume():
    if RESUME in ("none", "", "False", "false"):
        return None
    if RESUME != "auto":
        p = Path(RESUME)
        assert p.exists(), p
        if not is_valid_resume_dir(p):
            raise SystemExit(
                f"RESUME path invalid for trainer resume (need trainer_state.json; "
                f"not final/): {p}"
            )
        return str(p)
    # auto: Drive LATEST → local LATEST → newest checkpoint-*
    candidates = []
    for root in filter(None, [DRIVE_CKPT, OUT_DIR]):
        latest = read_latest(root)
        if latest is not None:
            candidates.append(("latest", latest))
        ckpts = sorted(
            Path(root).glob("checkpoint-*"), key=lambda q: q.stat().st_mtime
        )
        if ckpts:
            candidates.append(("ckpt", ckpts[-1]))
    for kind, c in candidates:
        if is_valid_resume_dir(c):
            print("Resume auto →", c)
            return str(c)
        if (
            c.is_dir()
            and (c / "adapter_config.json").is_file()
            and not (c / "trainer_state.json").is_file()
        ):
            raise SystemExit(
                f"Resume pointer is adapter-only (no trainer_state.json): {c}\\n"
                "Do not use final/ as resume_from_checkpoint. Use checkpoint-* "
                "or clear LATEST and start fresh."
            )
    print("Resume auto: fresh start")
    return None

class Phase2Callback(TrainerCallback):
    def __init__(self, state: Phase2State):
        self.ps = state
        self.t0 = time.time()
        self.last_t = self.t0
        self.last_save = self.t0
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                torch.cuda.reset_peak_memory_stats(i)

    def _vram(self):
        if not torch.cuda.is_available():
            return 0.0, 0.0
        num_gpus = torch.cuda.device_count()
        alloc = sum(torch.cuda.memory_allocated(i) for i in range(num_gpus)) / 1e9
        peak = sum(torch.cuda.max_memory_allocated(i) for i in range(num_gpus)) / 1e9
        return alloc, peak

    def on_step_end(self, args, state, control, **kwargs):
        self.ps.global_step = state.global_step
        world = max(int(os.environ.get("WORLD_SIZE", torch.cuda.device_count() if torch.cuda.is_available() else 1)), 1)
        self.ps.tok_approx += (
            args.per_device_train_batch_size
            * args.gradient_accumulation_steps
            * MAX_SEQ_LEN
            * world
        )
        now = time.time()
        if (now - self.last_t) >= 30 or state.global_step % max(args.logging_steps, 1) == 0:
            alloc, peak = self._vram()
            loss = "n/a"
            if state.log_history:
                v = state.log_history[-1].get("loss")
                if isinstance(v, (int, float)):
                    loss = f"{v:.4f}"
            print(
                f"step={state.global_step} loss={loss} "
                f"VRAM={alloc:.1f}G peak={peak:.1f}G "
                f"elapsed={(now - self.t0) / 60:.1f}m",
                flush=True,
            )
            self.last_t = now
        if (now - self.last_save) >= CKPT_MINUTES * 60:
            control.should_save = True
            self.last_save = now
            print(f"Timed checkpoint ({CKPT_MINUTES:.0f} min)", flush=True)
        if WALL_CLOCK_HOURS and WALL_CLOCK_HOURS > 0:
            elapsed_h = (now - self.t0) / 3600.0
            if elapsed_h >= WALL_CLOCK_HOURS:
                print(
                    f"Wall-clock limit {WALL_CLOCK_HOURS:.1f}h reached "
                    f"(elapsed {elapsed_h:.2f}h); saving and stopping before Colab kill.",
                    flush=True,
                )
                control.should_training_stop = True
                control.should_save = True
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        v = metrics.get("eval_loss")
        if isinstance(v, (int, float)):
            self.ps.last_eval_loss = float(v)
            if math.isnan(v) or math.isinf(v):
                print("WARNING: eval_loss is nan/inf — check masking / data.", flush=True)
            else:
                print(f"eval_loss={v:.4f} (step {state.global_step})", flush=True)
                if v < self.ps.best_eval_loss:
                    self.ps.best_eval_loss = float(v)
        return control

    def on_save(self, args, state, control, **kwargs):
        ckpt = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if ckpt.exists():
            (ckpt / "phase2_state.json").write_text(
                json.dumps(asdict(self.ps), indent=2)
            )
            write_latest(OUT_DIR, ckpt)
            mirror_to_drive(ckpt, update_latest=True)
        return control

phase_state = Phase2State()
resume_path = resolve_resume()
if resume_path:
    sf = Path(resume_path) / "phase2_state.json"
    if sf.exists():
        d = json.loads(sf.read_text())
        phase_state = Phase2State(
            **{k: d[k] for k in Phase2State.__dataclass_fields__ if k in d}
        )
        print("Restored phase2_state step=", phase_state.global_step)

# Optimizer: QLoRA-friendly (paged 8-bit Adam when bitsandbytes is available)
optim = "adamw_torch"
try:
    import bitsandbytes  # noqa: F401
    optim = "paged_adamw_8bit"
except Exception:
    try:
        from transformers.training_args import OptimizerNames
        names = {n.value if hasattr(n, "value") else str(n) for n in OptimizerNames}
        if "adamw_8bit" in names or "adamw_8bit" in str(names):
            optim = "adamw_8bit"
    except Exception:
        pass
print("optim:", optim)

# Precision & multi-GPU options
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

sft_kwargs = dict(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    bf16=use_bf16,
    fp16=use_fp16,
    optim=optim,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    dataset_num_proc=2,
    packing=PACKING,
    dataset_text_field="text",
    max_seq_length=None if not PACKING else MAX_SEQ_LEN,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=False,
    do_eval=True,
)
if SMOKE or MAX_STEPS is not None:
    sft_kwargs["max_steps"] = int(MAX_STEPS if MAX_STEPS is not None else 30)
    sft_kwargs["warmup_steps"] = 5
else:
    sft_kwargs["num_train_epochs"] = NUM_TRAIN_EPOCHS
    sft_kwargs["warmup_ratio"] = 0.03
    sft_kwargs["max_steps"] = -1

if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    sft_kwargs["ddp_find_unused_parameters"] = False

if HUB_ADAPTER_ID and HF_TOKEN:
    sft_kwargs.update(
        push_to_hub=True,
        hub_model_id=HUB_ADAPTER_ID,
        hub_strategy="every_save",
        hub_private_repo=True,
    )

def build_sft_config(kwargs):
    kwargs = dict(kwargs)
    if not kwargs.get("packing", False):
        kwargs["max_seq_length"] = None
        kwargs["max_length"] = None
    try:
        return SFTConfig(**kwargs)
    except TypeError:
        pass
    if "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
        try:
            return SFTConfig(**kwargs)
        except TypeError:
            pass
    if "max_seq_length" in kwargs:
        kwargs["max_length"] = kwargs.pop("max_seq_length")
        try:
            return SFTConfig(**kwargs)
        except TypeError:
            pass
    for k in (
        "packing", "dataset_text_field", "logging_first_step",
        "max_length", "max_seq_length", "do_eval", "hub_private_repo",
    ):
        kwargs.pop(k, None)
    return SFTConfig(**kwargs)

args = build_sft_config(sft_kwargs)
trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    callbacks=[Phase2Callback(phase_state)],
)
try:
    trainer = SFTTrainer(**trainer_kwargs)
except TypeError:
    trainer_kwargs.pop("processing_class", None)
    trainer_kwargs["tokenizer"] = tokenizer
    trainer = SFTTrainer(**trainer_kwargs)

# Assistant-only loss (Qwen ChatML markers — include trailing newline)
INSTRUCTION_PART = "<|im_start|>user\n"
RESPONSE_PART = "<|im_start|>assistant\n"
trainer = train_on_responses_only(
    trainer,
    instruction_part=INSTRUCTION_PART,
    response_part=RESPONSE_PART,
)

# ── Hard gates (abort before long train) ───────────────────────────────────
packing_on = bool(getattr(trainer.args, "packing", False))
assert (not packing_on) and (PACKING is False), (
    f"FATAL: packing must be False for Phase 2 SFT; trainer.args.packing={packing_on}"
)
print("packing OFF (required)")

# Label audit via train dataloader (post train_on_responses_only)
dl = trainer.get_train_dataloader()
batch = next(iter(dl))
assert "labels" in batch, "Train batch missing labels — cannot audit response mask"
labels_t = batch["labels"]
input_ids_t = batch["input_ids"]
n_rows = min(8, int(labels_t.shape[0]))
fracs = []
decoded_example = None
for i in range(n_rows):
    labs = labels_t[i].tolist()
    ids = input_ids_t[i].tolist()
    total = len(labs)
    trainable = sum(1 for x in labs if x != -100)
    frac = trainable / max(total, 1)
    fracs.append(frac)
    if decoded_example is None:
        pieces = [
            tokenizer.decode([tid]) if lab != -100 else " "
            for tid, lab in zip(ids, labs)
        ]
        decoded_example = "".join(pieces)

assert fracs, "Label audit produced no rows"
for frac in fracs:
    if frac <= 0.0:
        # Print token lengths to help debug truncation
        lengths = [len(tokenizer.encode(train_dataset[j]["text"])) for j in range(min(4, len(train_dataset)))]
        raise SystemExit(
            "FATAL: trainable labels fraction is 0 (all -100). "
            "Check train_on_responses_only markers "
            f"{INSTRUCTION_PART!r} / {RESPONSE_PART!r} and ChatML formatting. "
            f"Sample token lengths: {lengths}"
        )
    if not (0.01 < frac < 0.90):
        lengths = [len(tokenizer.encode(train_dataset[j]["text"])) for j in range(min(4, len(train_dataset)))]
        raise SystemExit(
            f"FATAL: trainable label fraction {frac:.3f} outside sane band (1%–90%). "
            f"Likely masking or truncation. Sample token lengths: {lengths}"
        )

print(
    f"Label audit OK over {len(fracs)} rows; trainable frac min/max="
    f"{min(fracs):.3f}/{max(fracs):.3f}"
)
print("Masked decode (spaces = ignored labels) head:")
print(repr((decoded_example or "")[:500]))
assert decoded_example and any(ch.strip() for ch in decoded_example), (
    "Masked decode empty — assistant span not visible"
)

print("resume_path=", resume_path)
print(
    "eval_steps", EVAL_STEPS, "save_steps", SAVE_STEPS,
    "CKPT_MINUTES", CKPT_MINUTES, "effective_batch", BATCH * ACCUM,
)


### Train


In [ ]:
# @title Show current memory stats + train
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if num_gpus > 1:
    total_max_mem, total_start_mem = 0.0, 0.0
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        start_mem = round(torch.cuda.max_memory_reserved(i) / 1024 / 1024 / 1024, 3)
        max_mem = round(props.total_memory / 1024 / 1024 / 1024, 3)
        total_max_mem += max_mem
        total_start_mem += start_mem
        print(f"GPU {i} = {props.name}. Max memory = {max_mem} GB. Reserved = {start_mem} GB.")
    print(f"Total GPUs = {num_gpus}. Total Max memory = {round(total_max_mem, 3)} GB. Reserved = {round(total_start_mem, 3)} GB.")
    start_gpu_memory = total_start_mem
    max_memory = total_max_mem
elif num_gpus == 1:
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
    print(f"{start_gpu_memory} GB of memory reserved.")
else:
    start_gpu_memory = 0.0
    max_memory = 1.0
    print("Using CPU")

try:
    trainer_stats = trainer.train(resume_from_checkpoint=resume_path)
except torch.cuda.OutOfMemoryError as e:
    raise SystemExit(
        "CUDA OOM during Phase 2 SFT. Cut BATCH (e.g. 4→2) or raise ACCUM to keep "
        f"effective batch ≈{BATCH * ACCUM}. Original error: {e}"
    ) from e


In [ ]:
# @title Show final memory and time stats
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if num_gpus > 0:
    used_memory = round(sum(torch.cuda.max_memory_reserved(i) for i in range(num_gpus)) / 1024 / 1024 / 1024, 3)
else:
    used_memory = 0.0
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max(max_memory, 0.001) * 100, 3)
lora_percentage = round(used_memory_for_lora / max(max_memory, 0.001) * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")
print("global_step≈", phase_state.global_step, "tok_approx≈", f"{phase_state.tok_approx:,}")
print("best_eval_loss", phase_state.best_eval_loss, "last_eval_loss", phase_state.last_eval_loss)


<a name="Inference"></a>
### Inference smoke (4 tasks)

Format/sanity only — not a quality claim. Always `add_generation_prompt=True`.


In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPTS = {
    "code_instruct": "You are a helpful coding assistant. Implement correct, clear solutions.",
    "code_review": (
        "You are a senior GitHub code reviewer. Find bugs, risks, and security issues. "
        "Be concise. If the code is fine, say so."
    ),
    "code_review_fix": "You apply GitHub review feedback and produce the corrected code.",
    "security": (
        "You are a defensive security assistant. Explain the weakness and provide a "
        "secure fix or remediation."
    ),
}

SMOKE_PROMPTS = {
    "code_instruct": "Write a Python function `is_palindrome(s: str) -> bool`.",
    "code_review": (
        "Language: python\nFile: util.py\n```diff\n"
        "@@ def divide(a, b):\n-    return a / b\n+    return a / b  # no zero check\n```"
    ),
    "code_review_fix": (
        "Before:\n```python\ndef divide(a, b):\n    return a / b\n```\n"
        "Reviewer: Guard against division by zero."
    ),
    "security": "Explain SQL injection risk for: cursor.execute(f\"SELECT * FROM t WHERE id={user_id}\")",
}

from transformers import TextStreamer

for task, user in SMOKE_PROMPTS.items():
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS[task]},
        {"role": "user", "content": user},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    device = getattr(model, "device", torch.device("cuda:0" if torch.cuda.is_available() else "cpu"))
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    print("\n" + "=" * 60)
    print("TASK:", task)
    print("=" * 60)
    streamer = TextStreamer(tokenizer, skip_prompt=True)
    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=128,
        use_cache=True,
    )


<a name="Save"></a>
### Save adapters (every session end)

Adapters only (`final/`) — not a full merge. `LATEST` stays on newest `checkpoint-*` for auto-resume. Run Cell 22 only after training is fully done.


In [ ]:
final_dir = OUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

# Mirror adapters to Drive but do NOT point LATEST at final/ (resume needs checkpoint-*)
mirror_to_drive(final_dir, update_latest=False)

# Ensure LATEST still prefers newest checkpoint-* if present
for root in filter(None, [OUT_DIR, DRIVE_CKPT]):
    ckpts = sorted(Path(root).glob("checkpoint-*"), key=lambda p: p.stat().st_mtime)
    if ckpts:
        write_latest(root, ckpts[-1])
        print("LATEST →", ckpts[-1])
    else:
        print("No checkpoint-* under", root, "— LATEST unchanged (do not use final/ for resume)")

meta = {
    "phase": "phase2_sft_qlora",
    "base_model_dir": str(MODEL_DIR),
    "base_hub": MODEL_HUB,
    "dataset": DATASET,
    "hub_adapter_id": HUB_ADAPTER_ID,
    "packing": False,
    "load_in_4bit": LOAD_IN_4BIT,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lr": LR,
    "max_seq_len": MAX_SEQ_LEN,
    "batch": BATCH,
    "accum": ACCUM,
    "smoke": SMOKE,
    "global_step": phase_state.global_step,
    "best_eval_loss": phase_state.best_eval_loss,
    "resume": RESUME,
    "ckpt_minutes": CKPT_MINUTES,
}
(final_dir / "phase2_meta.json").write_text(json.dumps(meta, indent=2))
(final_dir / "phase2_state.json").write_text(json.dumps(asdict(phase_state), indent=2))
print("Saved adapters →", final_dir)

if HUB_ADAPTER_ID and HF_TOKEN:
    model.push_to_hub(HUB_ADAPTER_ID, private=True, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_ADAPTER_ID, private=True, token=HF_TOKEN)
    print("Pushed adapters →", HUB_ADAPTER_ID)

print(
    "Reminder: multi-session resume uses Drive checkpoint-* via RESUME=auto. "
    "Set RUN_FINAL_MERGE=True and run Cell 22 only after training is fully done."
)


### Next session

1. Re-run **Install** + **Config** with `SMOKE = False`, `RESUME = "auto"`, same `OUT_DIR` / Drive paths.
2. **1h Drive mirrors** (`CKPT_MINUTES=60`) are the continuity mechanism across Colab reconnects.
3. After smoke: raise `BATCH` toward 4–8 using peak VRAM; keep effective batch ~32–64 via `ACCUM`.
4. If loss flat / eval rises: lower LR or raise LoRA `r`; if all-`-100` after an Unsloth bump: re-check response markers + `\n`.
5. When the full epoch (or target) is finished: set `RUN_FINAL_MERGE = True` and run **Cell 22 once**.

See `fine-tune/FINE_TUNE_DECISIONS.md` §6–7.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 22 — Optional final merge (LAST RUN ONLY)
# Merge Phase-2 SFT LoRA into the CPT-merged BF16 domain base → standalone
# BF16 instruct-capable weights. Skip until training is complete.
# Does not affect resume. Gated by RUN_FINAL_MERGE (default False).
# Pattern: fine-tune/merge_cpt_lora_colab.py (BF16 base, not 4-bit).
# ═══════════════════════════════════════════════════════════════════════════

if not RUN_FINAL_MERGE:
    print(
        "RUN_FINAL_MERGE=False — skipping Cell 22 final merge. "
        "Flip to True only after the last training session."
    )
else:
    from peft import PeftModel
    from unsloth.chat_templates import get_chat_template as _get_chat_template

    def _require_adapter_dir(adapter_dir: Path) -> None:
        if not adapter_dir.is_dir():
            raise SystemExit(f"Adapter dir missing: {adapter_dir}")
        cfg = adapter_dir / "adapter_config.json"
        weights = list(adapter_dir.glob("adapter_model*.safetensors")) + list(
            adapter_dir.glob("adapter_model.bin")
        )
        if not cfg.is_file():
            raise SystemExit(f"Missing adapter_config.json in {adapter_dir}")
        if not weights:
            raise SystemExit(
                f"Missing adapter weights in {adapter_dir} "
                "(expected adapter_model.safetensors or .bin)"
            )
        if adapter_dir.name.startswith("checkpoint-"):
            raise SystemExit(
                f"Refusing mid-train checkpoint as merge adapter: {adapter_dir}\n"
                "Prefer OUT_DIR/final (or Drive …/final). "
                "Override only by copying that checkpoint into a final/ path."
            )

    # Resolve adapter: Drive final → local final
    adapter_candidates = []
    if DRIVE_CKPT is not None:
        adapter_candidates.append(DRIVE_CKPT / "final")
    adapter_candidates.append(OUT_DIR / "final")
    adapter_dir = None
    for c in adapter_candidates:
        if (c / "adapter_config.json").is_file():
            adapter_dir = c
            break
    if adapter_dir is None:
        raise SystemExit(
            "No SFT adapter final/ found under Drive or OUT_DIR. "
            "Run Cell 20 save before Cell 22 merge."
        )
    _require_adapter_dir(adapter_dir)

    # Resolve CPT-merged BF16 base (same preference as Cell 07)
    if USE_DRIVE and local_model_complete(MODEL_DRIVE):
        merge_base = MODEL_DRIVE
    elif local_model_complete(MODEL_CACHE):
        merge_base = MODEL_CACHE
    elif local_model_complete(MODEL_DIR):
        merge_base = MODEL_DIR
    else:
        raise SystemExit(
            "CPT-merged base not found for BF16 merge. "
            "Restore Drive merge or Hub snapshot first."
        )

    out_dir = Path(MERGE_OUT_DIR)
    if out_dir.exists():
        if not FORCE_MERGE:
            raise SystemExit(
                f"Merge out_dir already exists: {out_dir}\n"
                "Set FORCE_MERGE=True to delete and rewrite, or change MERGE_OUT_DIR."
            )
        print(f"FORCE_MERGE: removing {out_dir}", flush=True)
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Free 4-bit train model before BF16 load (A100 40GB+)
    print("Freeing train model from GPU…", flush=True)
    try:
        del model
    except NameError:
        pass
    try:
        del trainer
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Base (BF16):", merge_base, flush=True)
    print("Adapter:    ", adapter_dir, flush=True)
    print("Out:        ", out_dir, flush=True)

    print("Loading CPT-merged base BF16 (load_in_4bit=False)…", flush=True)
    merge_model, merge_tok = FastLanguageModel.from_pretrained(
        model_name=str(merge_base),
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=False,
        token=HF_TOKEN,
    )
    merge_tok = _get_chat_template(merge_tok, chat_template="qwen-2.5")

    print("Attaching Phase-2 SFT LoRA…", flush=True)
    merge_model = PeftModel.from_pretrained(merge_model, str(adapter_dir))
    print("Merging + unloading…", flush=True)
    merge_model = merge_model.merge_and_unload()

    print("Saving merged BF16 weights…", flush=True)
    saved = False
    if hasattr(merge_model, "save_pretrained_merged"):
        try:
            merge_model.save_pretrained_merged(
                str(out_dir),
                merge_tok,
                save_method="merged_16bit",
            )
            saved = True
            print("Saved via Unsloth save_pretrained_merged(merged_16bit).", flush=True)
        except Exception as e:
            print(f"Unsloth merged save failed ({e}); falling back to HF save.", flush=True)

    if not saved:
        merge_model.save_pretrained(str(out_dir), safe_serialization=True)
        merge_tok.save_pretrained(str(out_dir))
        print("Saved via model.save_pretrained + tokenizer.", flush=True)

    # Ensure ChatML tokenizer is on disk for inference-ready artifact
    merge_tok.save_pretrained(str(out_dir))

    merge_meta = {
        "phase": "sft_merged_instruct_capable",
        "cpt_base": str(merge_base),
        "sft_adapter": str(adapter_dir),
        "out_dir": str(out_dir),
        "hub_merged_sft_id": HUB_MERGED_SFT_ID or None,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "dataset": DATASET,
        "notes": "Merged SFT adapters into CPT-merged BF16 base; ChatML tokenizer saved.",
    }
    (out_dir / "merge_meta.json").write_text(json.dumps(merge_meta, indent=2))

    has_cfg = (out_dir / "config.json").is_file()
    weight_files = list(out_dir.glob("*.safetensors")) + list(
        out_dir.glob("pytorch_model*.bin")
    )
    large = [p for p in weight_files if p.stat().st_size > 50_000_000]
    if not has_cfg or not large:
        raise SystemExit(
            f"Merge output looks incomplete under {out_dir} "
            f"(config.json={has_cfg}, large_weight_files={len(large)})."
        )
    gb = sum(p.stat().st_size for p in out_dir.rglob("*") if p.is_file()) / 1e9
    print(f"Merge complete ({gb:.1f} GB) → {out_dir}", flush=True)

    if HUB_MERGED_SFT_ID:
        if not HF_TOKEN:
            raise SystemExit("HUB_MERGED_SFT_ID set but no HF token")
        print(f"Pushing merged model → {HUB_MERGED_SFT_ID} (private)…", flush=True)
        merge_model.push_to_hub(HUB_MERGED_SFT_ID, private=True, token=HF_TOKEN)
        merge_tok.push_to_hub(HUB_MERGED_SFT_ID, private=True, token=HF_TOKEN)
        print("Hub push done.", flush=True)

    print("Phase 2 final merge done. Artifact is inference-ready (ChatML).", flush=True)


### GGUF Quantization & Export (Ollama / llama.cpp)
Converts the merged safetensors model (from HuggingFace or Drive/Local) into **GGUF** format across multiple quantization levels (`BF16`, `Q8_0`, `Q6_K`, `Q5_K_M`, `Q4_K_M`, `Q3_K_M`).
Saves GGUF files to Google Drive and uploads them to HuggingFace Hub.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 23 — Standalone GGUF Export & Quantization for Ollama / llama.cpp
# Self-reliant cell: Can be run in a fresh Colab/Kaggle session without running
# previous cells. Converts merged safetensors model (HF or Drive) to GGUF in:
#   BF16 / F16, Q8_0, Q6_K, Q5_K_M, Q4_K_M, Q3_K_M
# Saves GGUF files to Google Drive and uploads to HuggingFace Hub.
# ═══════════════════════════════════════════════════════════════════════════

import os, sys, gc, shutil, json, subprocess
from pathlib import Path

# ── 0. Auto-install required packages if running standalone ───────────────
try:
    import torch
    import unsloth
    from huggingface_hub import HfApi, create_repo
except ImportError:
    print("Installing required libraries for standalone GGUF export...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth", "huggingface_hub", "torch", "transformers", "accelerate"], check=False)
    import torch
    import unsloth
    from huggingface_hub import HfApi, create_repo

from unsloth import FastLanguageModel

# ── 1. User Configuration (Standalone Inputs) ─────────────────────────────
# Hugging Face Token (leave empty to load from Colab Secrets / Environment)
HF_TOKEN = ""

# Source Merged Model (safetensors format): Hugging Face repo ID OR Drive / local path
# Examples:
#   "Aniket200325/coder-qwen25-coder-7b-sft-qlora-v1-merged"
#   "/content/drive/MyDrive/coder-qwen25-coder-7b-sft-merged"
SOURCE_MODEL_ID_OR_PATH = (
    HUB_MERGED_SFT_ID
    if ("HUB_MERGED_SFT_ID" in globals() and HUB_MERGED_SFT_ID)
    else (
        str(MERGE_OUT_DIR)
        if ("MERGE_OUT_DIR" in globals() and Path(MERGE_OUT_DIR).exists())
        else "Aniket200325/coder-qwen25-coder-7b-sft-qlora-v1-merged"
    )
)

# Target Hugging Face Repo ID for GGUF Upload
TARGET_GGUF_HUB_REPO_ID = (
    (HUB_MERGED_SFT_ID + "-gguf")
    if ("HUB_MERGED_SFT_ID" in globals() and HUB_MERGED_SFT_ID)
    else "Aniket200325/coder-qwen25-coder-7b-sft-qlora-v1-gguf"
)

# Quantization Methods to Generate
# Options: "bf16", "f16", "q8_0", "q6_k", "q5_k_m", "q4_k_m", "q3_k_m"
QUANT_METHODS = ["bf16", "q8_0", "q6_k", "q5_k_m", "q4_k_m", "q3_k_m"]

# Save GGUF files to Google Drive?
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/coder-qwen25-coder-7b-sft-gguf")

# Local Working Directory for GGUF Generation
LOCAL_GGUF_DIR = Path("/content/gguf_export")
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir():
    LOCAL_GGUF_DIR = Path("/kaggle/working/gguf_export")
# ────────────────────────────────────────────────────────────────────────────

# ── 2. Environment & Auth Setup ───────────────────────────────────────────
IS_COLAB = "COLAB_" in "".join(os.environ.keys()) or Path("/content").is_dir()

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        HF_TOKEN = ""
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or ""

if SAVE_TO_DRIVE and IS_COLAB:
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            print("Mounting Google Drive...", flush=True)
            drive.mount("/content/drive")
        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"✓ Drive output target ready: {DRIVE_OUTPUT_DIR}", flush=True)
    except Exception as e:
        print(f"Notice: Google Drive unavailable ({e}). Continuing without Drive mirroring.")
        SAVE_TO_DRIVE = False

LOCAL_GGUF_DIR.mkdir(parents=True, exist_ok=True)

print("\n=== Standalone GGUF Export & Quantization ===")
print(f"Source Model:  {SOURCE_MODEL_ID_OR_PATH}")
print(f"Quant Methods: {QUANT_METHODS}")
print(f"HF Target Repo:{TARGET_GGUF_HUB_REPO_ID}")
print(f"Local Output:  {LOCAL_GGUF_DIR}")
print(f"Drive Target:  {DRIVE_OUTPUT_DIR if SAVE_TO_DRIVE else 'Disabled'}")

# Free VRAM & RAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── 3. Load Model & Tokenizer in 16-bit ───────────────────────────────────
print("\n1. Loading source model & tokenizer for GGUF conversion...", flush=True)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SOURCE_MODEL_ID_OR_PATH,
    max_seq_length=8192,
    dtype=None,
    load_in_4bit=False,  # Load in 16-bit for clean GGUF quantization
    token=HF_TOKEN or None,
)

# ── 4. Quantize to GGUF Formats ───────────────────────────────────────────
print("\n2. Quantizing and saving GGUF files locally...", flush=True)
for method in QUANT_METHODS:
    print(f"\n---> Exporting GGUF ({method})...", flush=True)
    try:
        model.save_pretrained_gguf(
            str(LOCAL_GGUF_DIR),
            tokenizer,
            quantization_method=method,
        )
        print(f"✓ Successfully exported GGUF ({method})", flush=True)
    except Exception as e:
        print(f"⚠ Warning: GGUF export for {method} failed: {e}", flush=True)

# List generated GGUF files
gguf_files = list(LOCAL_GGUF_DIR.glob("*.gguf"))
print(f"\n3. Generated {len(gguf_files)} GGUF file(s):", flush=True)
for p in gguf_files:
    size_gb = p.stat().st_size / (1024 ** 3)
    print(f"  - {p.name} ({size_gb:.2f} GB)")

# ── 5. Generate Ollama Modelfile ───────────────────────────────────────────
modelfile_content = """FROM ./unsloth.Q4_K_M.gguf
TEMPLATE """<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
"""
(LOCAL_GGUF_DIR / "Modelfile").write_text(modelfile_content)

# ── 6. Mirror Artifacts to Google Drive ────────────────────────────────────
if SAVE_TO_DRIVE and DRIVE_OUTPUT_DIR and gguf_files:
    print(f"\n4. Mirroring GGUF files & Modelfile to Google Drive...", flush=True)
    for p in list(LOCAL_GGUF_DIR.glob("*")):
        if p.is_file():
            dest = DRIVE_OUTPUT_DIR / p.name
            print(f"  Copying {p.name} -> Drive ({dest})...", flush=True)
            shutil.copy2(p, dest)
    print("✓ Google Drive GGUF mirror complete!", flush=True)

# ── 7. Upload to Hugging Face Hub ──────────────────────────────────────────
if TARGET_GGUF_HUB_REPO_ID and gguf_files:
    if not HF_TOKEN:
        print("\n⚠ Notice: HF_TOKEN not set; skipping Hugging Face Hub upload.")
    else:
        print(f"\n5. Uploading GGUF files to Hugging Face Hub ({TARGET_GGUF_HUB_REPO_ID})...", flush=True)
        try:
            api = HfApi(token=HF_TOKEN)
            create_repo(
                repo_id=TARGET_GGUF_HUB_REPO_ID,
                token=HF_TOKEN,
                private=False,
                exist_ok=True,
                repo_type="model",
            )
            api.upload_folder(
                folder_path=str(LOCAL_GGUF_DIR),
                repo_id=TARGET_GGUF_HUB_REPO_ID,
                repo_type="model",
            )
            print(f"✓ Hugging Face Hub upload complete -> https://huggingface.co/{TARGET_GGUF_HUB_REPO_ID}", flush=True)
        except Exception as e:
            print(f"⚠ Warning: Hugging Face Hub upload failed: {e}", flush=True)

print("\n=== Standalone GGUF Export & Upload Finished ===")
